# 4 · 和 Python 生态一起用

`pimeta` 不 import pandas / scikit-learn / xarray / networkx —— 它只**实现它们的协议**。
所以你想用哪个就用哪个，不装也能跑。

这个 notebook 演示五件事：

1. pandas DataFrame 直接喂进 `rma()`
2. `tidy()` / `glance()` 的结果进 pandas
3. `rma()` 的结果塞进 scikit-learn 的 `Pipeline` 和 `GridSearchCV`
4. 网络 meta 结果导出成 xarray `Dataset` 和 networkx `Graph`


In [1]:
import numpy as np
import pandas as pd

import pimeta as ma

In [2]:
study = ["Aronson 1948", "Ferguson & Simes 1949", "Rosenthal et al 1960",
         "Hart & Sutherland 1977", "Frimodt-Moller et al 1973",
         "Stein & Aronson 1953", "Vandiviere et al 1973", "TPT Madras 1980",
         "Coetzee & Berjak 1968", "Rosenthal et al 1961", "Comstock et al 1974",
         "Comstock & Webster 1969", "Comstock et al 1976"]
year = [1948, 1949, 1960, 1977, 1973, 1953, 1973, 1980, 1968, 1961, 1974, 1969, 1976]
tpos = [4, 6, 3, 62, 33, 180, 8, 505, 29, 17, 186, 5, 27]
tneg = [119, 300, 228, 13536, 5036, 1361, 2537, 87886, 7470, 1699, 50448, 2493, 16886]
cpos = [11, 29, 11, 248, 47, 372, 10, 499, 45, 65, 141, 3, 29]
cneg = [128, 274, 209, 12619, 5761, 1079, 619, 87892, 7232, 1600, 27197, 2338, 17825]

es = ma.escalc("RR", ai=tpos, bi=tneg, ci=cpos, di=cneg)
frame = pd.DataFrame({"study": study, "year": year, "yi": es.yi, "vi": es.vi})
frame.head()

,study,year,yi,vi
0,Aronson 1948,1948,-0.889311,0.325585
1,Ferguson & Simes 1949,1949,-1.585389,0.194581
2,Rosenthal et al 1960,1960,-1.348073,0.415368
3,Hart & Sutherland 1977,1977,-1.441551,0.020010
4,Frimodt-Moller et al 1973,1973,-0.217547,0.051210


## 1–2 · DataFrame 进，DataFrame 出

In [3]:
res = ma.rma("yi", "vi", mods="~ year", data=frame)

print("系数名:", res.coefnames)
print(ma.interop.tidy(res).to_string(index=False))

系数名: ['intrcpt', 'year']
   term   estimate  std.error  statistic  p.value    conf.low  conf.high
intrcpt -53.171118  32.011606  -1.660995 0.096714 -115.912713   9.570477
   year   0.026663   0.016269   1.638849 0.101245   -0.005224   0.058550


In [4]:
glance = ma.interop.glance(res)
row = pd.DataFrame({k: [v.iloc[0] if hasattr(v, "iloc") else v]
                    for k, v in glance.items()})
row[["k", "tau2", "I2", "QE", "QEp", "AIC"]].round(4)

,k,tau2,I2,QE,QEp,AIC
0,13,0.2666,87.6936,100.1782,0.0,26.9419


## 3 · 塞进 scikit-learn

`MetaRegressor` 实现了 `get_params` / `set_params` / `fit` / `predict` / `score`，
还带 `__sklearn_tags__`，所以 `is_regressor`、`clone`、`Pipeline`、`GridSearchCV`、
`cross_val_score` 全都能用。样本权重就是 `1/vi`（逆方差加权）。

In [5]:
from sklearn.base import clone, is_regressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = frame[["year"]].to_numpy(dtype=float)
y = frame["yi"].to_numpy(dtype=float)
w = 1.0 / frame["vi"].to_numpy(dtype=float)

est = ma.interop.MetaRegressor(method="REML")
print("is_regressor:", is_regressor(est))

pipe = Pipeline([("scale", StandardScaler()),
                 ("meta", ma.interop.MetaRegressor())])
grid = GridSearchCV(pipe, {"meta__method": ["DL", "REML", "PM"]},
                    cv=3, scoring="neg_mean_absolute_error")
grid.fit(X, y, meta__sample_weight=w)

print("最优参数:", grid.best_params_)
print("最优 MAE:", round(-grid.best_score_, 6))

is_regressor: True
最优参数: {'meta__method': 'PM'}
最优 MAE: 0.572838


## 4 · 网络结果导出成 xarray / networkx

（这里用一个 3 干预的小网络演示。）

In [6]:
nm = ma.netmeta(
    TE=[0.30, -0.20, 0.50, 0.10, -0.15, 0.25],
    seTE=[0.12, 0.15, 0.18, 0.14, 0.11, 0.13],
    treat1=["A", "B", "A", "C", "B", "C"],
    treat2=["B", "A", "C", "A", "C", "B"],
    studlab=["S1", "S2", "S3", "S4", "S5", "S6"], sm="RR",
)

ds = ma.interop.to_xarray(nm)
print(ds)

graph = ma.interop.to_networkx(nm)
print("节点:", list(graph.nodes))
print("边:", list(graph.edges(data=True))[:2])

<xarray.Dataset> Size: 2kB
Dimensions:               (treat1: 3, treat2: 3, treatment: 3)
Coordinates:
  * treat1                (treat1) <U1 12B 'A' 'B' 'C'
  * treat2                (treat2) <U1 12B 'A' 'B' 'C'
  * treatment             (treatment) <U1 12B 'A' 'B' 'C'
Data variables: (12/35)
    TE_common             (treat1, treat2) float64 72B nan 0.2788 ... 0.1774 nan
    seTE_common           (treat1, treat2) float64 72B 0.0 0.07766 ... 0.0
    lower_common          (treat1, treat2) float64 72B nan 0.1266 ... nan
    upper_common          (treat1, treat2) float64 72B nan 0.431 ... 0.3198 nan
    statistic_common      (treat1, treat2) float64 72B nan 3.59 ... 2.442 nan
    pval_common           (treat1, treat2) float64 72B nan 0.0003309 ... nan
    ...                    ...
    P_common              (treat1, treat2) float64 72B nan 0.0001655 ... nan
    P_random              (treat1, treat2) float64 72B nan 0.004134 ... nan
    Cov_common            (treat1, treat2) float64 72B 0

## 小结

| 你手上的东西 | 怎么接 |
|---|---|
| pandas / polars / arrow | 直接传给 `data=`，或用 `to_table(backend=...)` 拿走 |
| scikit-learn 工作流 | `ma.interop.MetaRegressor`（含 `__sklearn_tags__`） |
| statsmodels 报告 | `ma.interop.as_results(res)`，配 `summary_col` |
| xarray | `ma.interop.to_xarray(netmeta_result)` |
| networkx | `ma.interop.to_networkx(netmeta_result)` |

这些是**协议**不是适配器：没有装 pandas 也能用 pimeta，装了就能互操作。